In [0]:
from pyspark.sql import Row

employee_data = [
    Row(id=1, name="Joe", salary=85000, departmentId=1),
    Row(id=2, name="Henry", salary=80000, departmentId=2),
    Row(id=3, name="Sam", salary=60000, departmentId=2),
    Row(id=4, name="Max", salary=90000, departmentId=1),
    Row(id=5, name="Janet", salary=69000, departmentId=1),
    Row(id=6, name="Randy", salary=85000, departmentId=1),
    Row(id=7, name="Will", salary=70000, departmentId=1)
]

department_data = [
    Row(id=1, name="IT"),
    Row(id=2, name="Sales")
]

employee_df = spark.createDataFrame(employee_data)
department_df = spark.createDataFrame(department_data)

employee_df.createOrReplaceTempView("Employee")
department_df.createOrReplaceTempView("Department")

display(employee_df)
display(department_df)

In [0]:
"""
    A company's executives are interested in seeing who earns the most money in each of the company's departments. A high earner in a department is an employee who has a salary in the top three unique salaries for that department.

Write a solution to find the employees who are high earners in each of the departments.

Return the result table in any order.

The result format is in the following example.
"""

In [0]:
spark.sql("""
    WITH RankedEmployees AS (
        SELECT *,
        DENSE_RANK() OVER (PARTITION BY e.departmentId ORDER BY e.salary DESC) AS ranks_emp
        FROM Employee e
    )
    SELECT d.name AS department, e.name AS employee, e.salary AS salary
    FROM RankedEmployees e
    JOIN Department d ON e.departmentId = d.id
    WHERE e.ranks_emp <= 3
      
""").show()

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

ranked = employee_df.withColumn(
    'ranks_emp',
    F.dense_rank().over(Window.partitionBy(F.col('departmentId')).orderBy(F.col('salary').desc()))
)

ranked.alias('r').join(department_df.alias('d'), F.col('r.departmentId') == F.col('d.id')).filter(F.col('ranks_emp') <= 3).show()